# Django Forms

## Why Django Forms?

Django Forms handle three tasks together:
1. **Rendering** — generate HTML `<input>` elements.
2. **Validation** — check data and produce helpful error messages.
3. **Cleaning** — convert raw strings to Python objects (e.g. `int`, `date`).

Without forms you must write all three steps manually for every view.


## forms.Form — Manual Form

```python
# profiles/forms.py
from django import forms

class ContactForm(forms.Form):
    name    = forms.CharField(max_length=100)
    email   = forms.EmailField()
    message = forms.CharField(widget=forms.Textarea)
    age     = forms.IntegerField(required=False, min_value=0, max_value=120)
```

Common field types:

| Class | Validates / Produces |
|-------|---------------------|
| `CharField` | Non-empty string |
| `EmailField` | Valid e-mail address |
| `IntegerField` | Integer within optional min/max |
| `DateField` | `datetime.date` |
| `BooleanField` | `True` / `False` |
| `ChoiceField` | Value from a set of choices |
| `FileField` | Uploaded file |


## Processing a Form in a View

```python
# profiles/views.py
from django.shortcuts import render, redirect
from django.contrib import messages
from .forms import ContactForm

def contact_view(request):
    if request.method == 'POST':
        form = ContactForm(request.POST)
        if form.is_valid():
            name    = form.cleaned_data['name']
            email   = form.cleaned_data['email']
            message = form.cleaned_data['message']
            # do something with the data ...
            messages.success(request, f'Message from {name} received.')
            return redirect('contact')
    else:
        form = ContactForm()
    return render(request, 'contact.html', {'form': form})
```

`form.is_valid()` runs all validators and populates `form.cleaned_data` on success, or `form.errors` on failure.


## Rendering Forms in Templates

```html
<!-- templates/contact.html -->
<form method="post">
    {% csrf_token %}
    {{ form.as_p }}
    <button type="submit">Send</button>
</form>
```

`{% csrf_token %}` inserts the hidden CSRF field that Django requires for all POST requests.

Render helpers:

| Helper | Output |
|--------|--------|
| `{{ form.as_p }}` | Each field wrapped in `<p>` |
| `{{ form.as_ul }}` | Each field as `<li>` |
| `{{ form.as_table }}` | Each field as `<tr>` |
| `{{ form.field_name }}` | A single field widget |
| `{{ form.field_name.errors }}` | Errors for one field |
| `{{ form.errors }}` | All errors |


## ModelForm — Form from a Model

`ModelForm` automatically generates form fields from a model's field definitions.

```python
# profiles/forms.py
from django import forms
from .models import Profile

class ProfileForm(forms.ModelForm):
    class Meta:
        model  = Profile
        fields = ['bio', 'avatar']          # or: exclude = ['user']
        widgets = {
            'bio': forms.Textarea(attrs={'rows': 4}),
        }
        labels  = {'bio': 'About you'}
        help_texts = {'avatar': 'Upload a profile picture (optional).'}
```

```python
# views.py
def profile_edit(request):
    profile = request.user.profile
    if request.method == 'POST':
        form = ProfileForm(request.POST, request.FILES, instance=profile)
        if form.is_valid():
            form.save()                     # updates the existing instance
            return redirect('profile')
    else:
        form = ProfileForm(instance=profile)
    return render(request, 'profile_edit.html', {'form': form})
```

`instance=` pre-populates the form with existing data and tells `save()` to UPDATE instead of INSERT.


## Custom Validation

### Field-level validator
```python
class ContactForm(forms.Form):
    phone = forms.CharField(max_length=15)

    def validate_phone(value):
        if not value.startswith('+'):
            raise forms.ValidationError("Phone must start with '+'.")

    phone = forms.CharField(max_length=15, validators=[validate_phone])
```

### clean_<field> method
```python
class ContactForm(forms.Form):
    name = forms.CharField(max_length=100)

    def clean_name(self):
        value = self.cleaned_data['name']
        if value.lower() == 'admin':
            raise forms.ValidationError("'admin' is not allowed as a name.")
        return value.strip()
```

### Object-level (cross-field) validation
```python
class PasswordChangeForm(forms.Form):
    password  = forms.CharField(widget=forms.PasswordInput)
    password2 = forms.CharField(widget=forms.PasswordInput, label='Confirm password')

    def clean(self):
        cleaned = super().clean()
        p1 = cleaned.get('password')
        p2 = cleaned.get('password2')
        if p1 and p2 and p1 != p2:
            raise forms.ValidationError("Passwords do not match.")
        return cleaned
```


## The Messages Framework

Django's messages framework sends one-time notifications that persist across a redirect.

```python
# settings.py
INSTALLED_APPS += ['django.contrib.messages']
MIDDLEWARE += ['django.contrib.messages.middleware.MessageMiddleware']
MESSAGE_STORAGE = 'django.contrib.messages.storage.session.SessionStorage'
```

```python
from django.contrib import messages

def contact_view(request):
    if request.method == 'POST':
        form = ContactForm(request.POST)
        if form.is_valid():
            # process ...
            messages.success(request, 'Your message was sent!')
            return redirect('contact')
        else:
            messages.error(request, 'Please fix the errors below.')
    else:
        form = ContactForm()
    return render(request, 'contact.html', {'form': form})
```

Display in the template:
```html
{% for message in messages %}
    <div class="alert alert-{{ message.tags }}">{{ message }}</div>
{% endfor %}
```

Message levels: `debug`, `info`, `success`, `warning`, `error`.


## CSRF Protection

Django rejects POST requests that lack a valid CSRF token. How it works:

1. On the first GET, Django sets a cookie (`csrftoken`).
2. The template tag `{% csrf_token %}` injects a matching hidden field.
3. On POST, Django checks that the hidden field value matches the cookie.

For AJAX POST requests, include the token in the header:
```javascript
fetch('/contact/', {
    method: 'POST',
    headers: {
        'Content-Type': 'application/json',
        'X-CSRFToken': document.cookie.match(/csrftoken=([^;]+)/)[1],
    },
    body: JSON.stringify({name: 'Alice'}),
});
```

To exempt a specific view (use with care):
```python
from django.views.decorators.csrf import csrf_exempt

@csrf_exempt
def webhook(request):
    ...
```


## Summary

- `forms.Form` defines fields, validates input, and provides `cleaned_data`.
- `forms.ModelForm` auto-generates fields from a model; `save()` creates or updates the instance.
- `form.is_valid()` runs all validators; use `form.errors` when it returns `False`.
- Use `clean_<field>()` for field-level logic and `clean()` for cross-field logic.
- `{% csrf_token %}` is required in every HTML form that issues a POST request.
- The messages framework stores one-time notifications in the session and displays them after a redirect.
- Pass `request.FILES` when the form handles file uploads (`enctype="multipart/form-data"` on the `<form>` tag).
